https://pubmed.ncbi.nlm.nih.gov/39962516/


## Depression (Vu et al., 2025)
**Reference**: Vu et al. "Prediction of depressive disorder using machine learning 
approaches: findings from the NHANES." BMC Medical Informatics and Decision Making, 2025.

**Label**: PHQ-9 score ≥ 10 (moderate-to-severe depression)

**Inclusion criteria**: Adults ≥ 18 years, NHANES 2013–2014

**Features**: Gender, age, race/ethnicity, BMI, education, marital status, 
poverty-income ratio, serum cotinine, hypertension history, fasting glucose, eGFR 
(computed from serum creatinine using CKD-EPI equation)

 **Deviations from reference study**:
- Reference restricted to NHANES 2013–2014 (n≈5,000); we include all cycles 
  with PHQ-9 data (2005–2018), yielding n=44,535
- Reference uses 80/20 train/test split; we use 5-fold stratified cross-validation
- eGFR computed identically to reference (CKD-EPI 2021)

In [8]:
import pandas as pd
import numpy as np
import pickle

full = pd.read_pickle("/Users/alva/Documents/nhanes_tasks/data/NHANES/NHANES_master.pkl")

def calc_egfr(row):
    scr = row['LBXSCR']
    age = row['RIDAGEYR']
    sex = row['RIAGENDR']
    if pd.isna(scr) or pd.isna(age) or pd.isna(sex):
        return np.nan
    kappa = 0.7 if sex == 2 else 0.9
    alpha = -0.241 if sex == 2 else -0.302
    sex_factor = 1.012 if sex == 2 else 1.0
    ratio = scr / kappa
    return 142 * (min(ratio, 1)**alpha) * (max(ratio, 1)**-1.200) * (0.9938**age) * sex_factor

In [9]:
phq_cols = ['DPQ010','DPQ020','DPQ030','DPQ040','DPQ050','DPQ060','DPQ070','DPQ080','DPQ090']

subset_dep = full[full['RIDAGEYR'] >= 18].copy()
subset_dep['PHQ9'] = subset_dep[phq_cols].sum(axis=1, skipna=False)
subset_dep = subset_dep[subset_dep[phq_cols].notna().all(axis=1)]
subset_dep['eGFR'] = subset_dep.apply(calc_egfr, axis=1)
subset_dep['depression'] = (subset_dep['PHQ9'] >= 10).astype(int)

features_depression = [
    'RIAGENDR', 'RIDAGEYR', 'RIDRETH1', 'BMXBMI', 'DMDEDUC2',
    'DMDMARTL', 'INDFMPIR', 'LBXCOT', 'BPQ020', 'LBXSGL', 'eGFR'
]

data_depression = subset_dep[features_depression + ['depression']].dropna(subset=['depression'])
X_depression = data_depression[features_depression].values
y_depression = data_depression['depression'].values

print(f"Depression: n={len(y_depression)}, prevalence={y_depression.mean():.3f}, features={X_depression.shape[1]}")

Depression: n=44535, prevalence=0.088, features=11


In [12]:
import os
os.makedirs('/Users/alva/Documents/nhanes_tasks/data/tasks/', exist_ok=True)
task = {
    'X': X_depression,
    'y': y_depression,
    'features': features_depression,
    'name': 'depression'
}

with open('/Users/alva/Documents/nhanes_tasks/data/tasks/depression.pkl', 'wb') as f:
    pickle.dump(task, f)

print("Saved.")

Saved.
